# Language Modeling

언어 모델은 앞에 나온 단어들을 보고 다음에 올 단어를 예측하는 모델이다. 예를 들어 `오늘 날씨가`라는 문맥이 주어졌을 때, 다음 단어로 `좋다`, `춥다`, `맑다` 같은 단어가 올 가능성을 계산한다.

이처럼 다음 단어를 예측할 수 있으면 문장 자동완성, 검색어 추천, 챗봇 응답 생성, 번역, 요약 같은 다양한 자연어 처리 작업으로 확장할 수 있다.

## 이미 학습된 언어 모델 체험

직접 언어 모델을 처음부터 학습하려면 많은 데이터와 시간이 필요하다. 그래서 여기서는 Hugging Face의 `transformers` 라이브러리에서 제공하는 GPT-2 모델을 불러와서 사용한다.

GPT-2는 앞 문맥을 보고 다음 토큰을 반복적으로 예측하여 문장을 생성하는 언어 모델이다.

> GPT-2 내부 구조는 Transformer 기반이지만, 이 파일에서는 구조를 깊게 다루지 않는다. 이후 Transformer 챕터에서 다시 연결해서 학습한다.

In [ ]:
# %pip install transformers

In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# 사용할 사전 학습 모델 이름
model_name = 'gpt2'

# 생성 모델에서 패딩이 필요한 경우 왼쪽에 패딩을 붙이도록 설정하여 토크나이저 생성
tokenizer = GPT2Tokenizer.from_pretrained(model_name,padding_side = 'left')

# 다음 토큰을 예측하는 사전 학습 언어 모델
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT2는 기본 pad_token이 없으므로 eos_token을 pad_token처럼 사용하도록 지정
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 입력 문장을 토큰으로 변환

In [2]:
# 입력 텍스트
input_text = 'The future of AI is'

# tokenizer는 텍스트를 모델이 처리할 수 있는 숫자 형태로 변환
inputs = tokenizer(
    input_text,
    return_tensors = 'pt',
    return_attention_mask = True
)

print(inputs)

{'input_ids': tensor([[ 464, 2003,  286, 9552,  318]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}


In [3]:
# 정수 id를 다시 토큰 문자열로 변환해서 확인
ids = inputs['input_ids'][0]
tokens = tokenizer.convert_ids_to_tokens(ids)

print('토큰 : ',tokens)

토큰 :  ['The', 'Ġfuture', 'Ġof', 'ĠAI', 'Ġis']


## 텍스트 생성

In [ ]:
output = model.generate(
    input_ids = inputs['input_ids'],
    attention_mask = inputs['attention_mask'],
    pad_token_id=tokenizer.eos_token_id,
    max_length = 50,
    num_return_sequences = 1,        # 생성할 문장의 갯수
    do_sample = True,                # 샘플링 여부, False로 생성할 경우 가장 확률이 높은 토큰을 선택하여 결과가 더 고정적
    temperature = 0.8,               # 생성의 무작위성 조절, 값이 낮으면 안정적이고 반복적 문장, 값이 높으면 다양하지만 어색한 문장
    top_k = 50,                      # 다음 토큰 후보 상위 50개 제한
    top_p = 0.95                     # 확률이 낮은 후보는 제외하여 생성 품질 안정화
)

print(output)

tensor([[  464,  2003,   286,  9552,   318,   991,   257, 10715,    13,   887,
           262, 12779,   389, 13079,    13,  1002,   674,  2003,   318,   355,
         10084,   290,  8925,   355, 16903,    11,   788,   356,   815,   307,
          1498,   284,  1205,   262,   845,  4899,   356,   765,   284,   307,
          1498,   284,   779,   287,   674, 10908,  3160,    13,   383,  2003]])


In [10]:
# 생성된 정수 토큰을 사람이 읽을 수 있는 문자열로 변환
result = tokenizer.decode(output[0],skip_special_tokens=True)   # 특수 토큰 출력에서 제거
print(result)

The future of AI is still a mystery. But the possibilities are endless. If our future is as diverse and dynamic as ours, then we should be able to develop the very tools we want to be able to use in our everyday lives. The future


## 여러 문장 생성해보기

In [13]:
input_text = "Natural language processing helps computers"
inputs = tokenizer(input_text, return_tensors='pt', return_attention_mask=True)

outputs = model.generate(
    input_ids=inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    pad_token_id=tokenizer.eos_token_id,
    max_length=50,
    num_return_sequences=3,          # 생성할 문장의 갯수
    do_sample=True,                  # 샘플링 여부, False로 설정할 경우 가장 확률이 높은 토큰을 선택하여 결과가 더 고정적
    temperature=0.5,                 # 생성의 무작위성 조절, 값이 낮으면 안정적이고 반복적 문장, 값이 높으면 다양하지만 어색한 문장
    top_k=50,                        # 다음 토큰 후보 상위 50개 제한
    top_p=0.95                       # 확률이 낮은 후보는 제외하여 생성 품질 안정화
)

for i, output in enumerate(outputs, start=1):
    print(f'[{i}]')
    print(tokenizer.decode(output, skip_special_tokens=True))
    print()

[1]
Natural language processing helps computers learn and understand language.

What is the difference between learning and memorization?

Learning is a process of learning to use and learn new information.

Mere memorization is learning to memorize a piece

[2]
Natural language processing helps computers learn new words and phrases.

The new language processing system is based on a computer's ability to learn and use new words.

The computer's ability to learn and use new words and phrases.

The

[3]
Natural language processing helps computers understand the meaning of words.

"This is the first time we have been able to use this kind of data to create a language that is as good as it can be," said Dr. David W. Johnson,

